# MCMCTree B() Calibrations to BEAST2 MRCAPrior Converter

Reads `strategy1.tree` (MCMCTree format with `B(min,max,p,c)` uniform prior annotations) and generates BEAST2 XML blocks for taxon sets and MRCAPrior distributions.

In [1]:
import re
from pathlib import Path

TREE_FILE = Path("strategy1.tree")
OUTPUT_FILE = Path("strategy1_mrca_priors.xml")

# Replace with your BEAST tree ID (e.g., "@Tree.t:Comb-16taxa-1x")
TREE_REF = "@Tree.t:alignment"

# Starting index for Uniform distribution IDs (adjust to avoid collisions)
UNIFORM_ID_START = 0

In [2]:
def parse_newick_with_calibrations(newick_str):
    """Parse a Newick string with 'B(min,max,p,c)' annotations.
    
    Returns:
        root: root node ID
        children: dict mapping node_id -> list of child node_ids
        leaf_name: dict mapping leaf node_id -> taxon name
        calibrations: dict mapping node_id -> (lower, upper)
    """
    s = newick_str.strip().rstrip(";")
    node_counter = 0
    children = {}
    leaf_name = {}
    calibrations = {}
    stack = []
    i = 0

    def new_node():
        nonlocal node_counter
        nid = node_counter
        node_counter += 1
        children[nid] = []
        return nid

    # Create root and skip opening '('
    root = new_node()
    stack.append(root)
    i = 1  # skip first '('

    while i < len(s):
        c = s[i]

        if c == "(":
            node = new_node()
            children[stack[-1]].append(node)
            stack.append(node)
            i += 1

        elif c == ",":
            i += 1

        elif c == ")":
            closed = stack.pop()
            i += 1
            # Check for 'B(...)' annotation
            if i < len(s) and s[i] == "'":
                end_quote = s.index("'", i + 1)
                annotation = s[i + 1 : end_quote]
                m = re.match(r"B\(([\d.]+),([\d.]+)", annotation)
                if m:
                    calibrations[closed] = (m.group(1), m.group(2))
                i = end_quote + 1
            # Skip optional branch length after ')' or after annotation
            if i < len(s) and s[i] == ":":
                i += 1
                while i < len(s) and s[i] not in "(),';":
                    i += 1

        elif c == ":":
            # Skip branch length
            i += 1
            while i < len(s) and s[i] not in "(),';":
                i += 1

        else:
            # Taxon name
            j = i
            while j < len(s) and s[j] not in "(),':\"":
                j += 1
            name = s[i:j].strip()
            if name:
                leaf = new_node()
                leaf_name[leaf] = name
                children[stack[-1]].append(leaf)
            i = j

    return root, children, leaf_name, calibrations


def get_leaves(node, children, leaf_name):
    """Recursively get all leaf taxon names under a node."""
    if node in leaf_name:
        return [leaf_name[node]]
    result = []
    for child in children[node]:
        result.extend(get_leaves(child, children, leaf_name))
    return result

In [3]:
# Parse the tree
tree_str = TREE_FILE.read_text()
root, children, leaf_name, calibrations = parse_newick_with_calibrations(tree_str)

all_taxa = sorted(set(leaf_name.values()))
print(f"Total taxa: {len(all_taxa)}")
print(f"Total calibrations: {len(calibrations)}")
print()

# Build calibration table: (node_id, clade_name, lower, upper, taxa_list)
calib_table = []
for node_id, (lower, upper) in calibrations.items():
    taxa = sorted(get_leaves(node_id, children, leaf_name))
    if len(taxa) == len(all_taxa):
        clade_name = "root_all"
    else:
        clade_name = f"{taxa[0]}_to_{taxa[-1]}"
    calib_table.append((node_id, clade_name, lower, upper, taxa))

# Ensure unique clade names
name_counts = {}
for i, (nid, name, lo, hi, taxa) in enumerate(calib_table):
    name_counts[name] = name_counts.get(name, 0) + 1
    if name_counts[name] > 1:
        calib_table[i] = (nid, f"{name}_{name_counts[name]}", lo, hi, taxa)

# Fix first occurrence too if duplicates exist
seen_names = {}
for i, (nid, name, lo, hi, taxa) in enumerate(calib_table):
    base = name.rsplit("_", 1)[0] if re.search(r'_\d+$', name) else name
    if base not in seen_names:
        seen_names[base] = []
    seen_names[base].append(i)

for base, indices in seen_names.items():
    if len(indices) > 1:
        for count, idx in enumerate(indices, 1):
            nid, _, lo, hi, taxa = calib_table[idx]
            calib_table[idx] = (nid, f"{base}_{count}", lo, hi, taxa)

# Print summary
print(f"{'Clade Name':<45} {'Lower':>8} {'Upper':>8} {'#Taxa':>6}")
print("-" * 70)
for nid, name, lo, hi, taxa in calib_table:
    print(f"{name:<45} {lo:>8} {hi:>8} {len(taxa):>6}")

Total taxa: 54
Total calibrations: 34

Clade Name                                       Lower    Upper  #Taxa
----------------------------------------------------------------------
Acropora_m_to_Nematostel                          5.29    6.361      5
Homo_sapie_to_Mus_muscul                         0.616    1.646      2
Homo_sapie_to_Ornithorhy                         1.649    2.015      3
Gallus_gal_to_Ornithorhy                          3.18    3.329      4
Gallus_gal_to_Xenopus_tr                          3.37     3.51      5
Danio_reri_to_Xenopus_tr_1                       4.207    4.537      6
Danio_reri_to_Xenopus_tr_2                       4.207    4.684      7
Eptatretus_to_Petromyzon                         3.585    6.361      2
Danio_reri_to_Xenopus_tr_3                       4.575    6.361      9
Ciona_inte_to_Xenopus_tr                          5.14    6.361     11
Branchiost_to_Xenopus_tr                          5.14    6.361     12
Ptychodera_to_Saccogloss              

In [4]:
# Print descendant taxa for each calibrated clade (for verification)
for nid, name, lo, hi, taxa in calib_table:
    print(f"\n{name} [B({lo},{hi})]:")
    for t in taxa:
        print(f"  {t}")


Acropora_m_to_Nematostel [B(5.29,6.361)]:
  Acropora_m
  Anemonia_v
  Hydra_magn
  Hydractini
  Nematostel

Homo_sapie_to_Mus_muscul [B(0.616,1.646)]:
  Homo_sapie
  Mus_muscul

Homo_sapie_to_Ornithorhy [B(1.649,2.015)]:
  Homo_sapie
  Mus_muscul
  Ornithorhy

Gallus_gal_to_Ornithorhy [B(3.18,3.329)]:
  Gallus_gal
  Homo_sapie
  Mus_muscul
  Ornithorhy

Gallus_gal_to_Xenopus_tr [B(3.37,3.51)]:
  Gallus_gal
  Homo_sapie
  Mus_muscul
  Ornithorhy
  Xenopus_tr

Danio_reri_to_Xenopus_tr_1 [B(4.207,4.537)]:
  Danio_reri
  Gallus_gal
  Homo_sapie
  Mus_muscul
  Ornithorhy
  Xenopus_tr

Danio_reri_to_Xenopus_tr_2 [B(4.207,4.684)]:
  Danio_reri
  Gallus_gal
  Homo_sapie
  Leucoraja_
  Mus_muscul
  Ornithorhy
  Xenopus_tr

Eptatretus_to_Petromyzon [B(3.585,6.361)]:
  Eptatretus
  Petromyzon

Danio_reri_to_Xenopus_tr_3 [B(4.575,6.361)]:
  Danio_reri
  Eptatretus
  Gallus_gal
  Homo_sapie
  Leucoraja_
  Mus_muscul
  Ornithorhy
  Petromyzon
  Xenopus_tr

Ciona_inte_to_Xenopus_tr [B(5.14,6.361)]:


In [5]:
def generate_mrca_prior_xml(calib_table, tree_ref, uniform_start=0):
    """Generate BEAST2 MRCAPrior XML blocks.
    
    Returns:
        prior_xml: MRCAPrior distribution blocks
        log_xml: log idref entries for trace logger
    """
    seen_taxa = set()
    prior_lines = []
    log_lines = []
    uid = uniform_start

    for nid, clade_name, lower, upper, taxa in calib_table:
        prior_lines.append(
            f'                <distribution id="{clade_name}.prior" '
            f'spec="beast.base.evolution.tree.MRCAPrior" '
            f'monophyletic="true" tree="{tree_ref}">'
        )
        prior_lines.append(
            f'                    <taxonset id="{clade_name}" spec="TaxonSet">'
        )

        for taxon in taxa:
            if taxon not in seen_taxa:
                prior_lines.append(
                    f'                        <taxon id="{taxon}" spec="Taxon"/>'
                )
                seen_taxa.add(taxon)
            else:
                prior_lines.append(
                    f'                        <taxon idref="{taxon}"/>'
                )

        prior_lines.append("                    </taxonset>")
        prior_lines.append(
            f'                    <Uniform id="Uniform.{uid}" '
            f'lower="{lower}" name="distr" upper="{upper}"/>'
        )
        prior_lines.append("                </distribution>")
        prior_lines.append("")

        log_lines.append(f'            <log idref="{clade_name}.prior"/>')
        uid += 1

    return "\n".join(prior_lines), "\n".join(log_lines)


prior_xml, log_xml = generate_mrca_prior_xml(calib_table, TREE_REF, UNIFORM_ID_START)

print("=" * 80)
print("MRCA PRIOR BLOCKS (paste inside <distribution id='prior' ...> block)")
print("=" * 80)
print(prior_xml)
print()
print("=" * 80)
print("LOG ENTRIES (paste inside <logger id='tracelog' ...> block)")
print("=" * 80)
print(log_xml)

MRCA PRIOR BLOCKS (paste inside <distribution id='prior' ...> block)
                <distribution id="Acropora_m_to_Nematostel.prior" spec="beast.base.evolution.tree.MRCAPrior" monophyletic="true" tree="@Tree.t:alignment">
                    <taxonset id="Acropora_m_to_Nematostel" spec="TaxonSet">
                        <taxon id="Acropora_m" spec="Taxon"/>
                        <taxon id="Anemonia_v" spec="Taxon"/>
                        <taxon id="Hydra_magn" spec="Taxon"/>
                        <taxon id="Hydractini" spec="Taxon"/>
                        <taxon id="Nematostel" spec="Taxon"/>
                    </taxonset>
                    <Uniform id="Uniform.0" lower="5.29" name="distr" upper="6.361"/>
                </distribution>

                <distribution id="Homo_sapie_to_Mus_muscul.prior" spec="beast.base.evolution.tree.MRCAPrior" monophyletic="true" tree="@Tree.t:alignment">
                    <taxonset id="Homo_sapie_to_Mus_muscul" spec="TaxonSet">
      

In [6]:
# Save to file
output = []
output.append("<!-- MRCA Prior blocks -->")
output.append("<!-- Paste inside <distribution id='prior' ...> block -->")
output.append(prior_xml)
output.append("")
output.append("<!-- Log entries -->")
output.append("<!-- Paste inside <logger id='tracelog' ...> block -->")
output.append(log_xml)

OUTPUT_FILE.write_text("\n".join(output), encoding="utf-8")
print(f"Saved to: {OUTPUT_FILE.resolve()}")

Saved to: /home/piyumal/PHD/TimeTree/IQ2MC/ESS_analysis/BEAST_analysis/Fossil_def_beast/strategy1_mrca_priors.xml
